# 07. Dataset variants: groupmate only, yours only, combined

Runs selection (02), freezing and testing (03) once per development dataset:

| Variant | Development parcels used |
|---|---|
| `groupmate` | parcels in the groupmate's package (`shared` + `groupmate_only`) |
| `yours` | parcels in your own collection before the merge (`shared` + `yours_only`) |
| `combined` | both |

Only whole development (train/validation) parcel groups are dropped. Locked partitions, labels,
quality rules and the external test parcels are the same for every variant, so test scores are
directly comparable. Each variant publishes its own `current_<stage>__<variant>.json` pointers;
`current_models.json` (used by the Paoay and Atok maps) is not changed.

**Input:** current inputs (01), the origins stage. **Output:** `current_{selection,models,test}__<variant>.json`.
Each variant takes about as long as running 02 and 03.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
from IPython.display import display
from model_aza.improved import datasets, workflow
pd.set_option("display.width", 200)

## Origins

Built once from the groupmate's share package and your pre-merge campaign backup, then sealed, so later
runs do not need either folder. Set `rebuild = True` only if one of them changed.

In [ ]:
from pathlib import Path as _Path

groupmate_package = _Path.home() / "Downloads/groupmate_data_latest"
your_campaign = root / "sen4agrinet_work/adaptive_training_backup_20261119_before_groupmate_dataset"
rebuild = False

if rebuild or not (root / "pipeline_outputs/improved/current_origins.json").is_file():
    datasets.build_origins(root, groupmate_package, your_campaign)
origins_receipt, origins = datasets.load_origins(root)
print("Origins receipt:", origins_receipt)
pd.Series(origins).value_counts().rename("parcel groups").to_frame()

## Development parcels per variant (current inputs)

In [ ]:
from model_aza.pilots import data

inputs_receipt, _ = workflow._input(root, None, "inputs")
rows, _ = data.load(inputs_receipt.parent)
table = pd.DataFrame(datasets.summary(rows, origins))
display(table.pivot_table(index=["variant", "partition"], columns="crop", values="parcels", aggfunc="sum", margins=True))

## Select, freeze and test each variant

In [ ]:
variants = ["groupmate", "yours", "combined"]
results = {}
for variant in variants:
    print(f"=== {variant}", flush=True)
    workflow.select(root, progress=lambda m: print(" ", m, flush=True), variant=variant)
    models = workflow.freeze(root, variant=variant)
    tested = workflow.test(root, variant=variant)
    results[variant] = {"model": json.loads((models.parent / "model.json").read_text()),
                        "scores": json.loads((tested.parent / "test_scores.json").read_text())}
    print("  test receipt:", tested, flush=True)

## Comparison

In [ ]:
records = []
for variant, r in results.items():
    for population, p in r["scores"]["populations"].items():
        records.append({"variant": variant, "candidate": r["scores"]["candidate"],
                        "development_parcels": len(r["model"]["development_groups"]),
                        "population": population, "parcels": p["parcels"],
                        "macro_f1": p["metrics"]["macro_f1"], "present_crop_macro_f1": p["metrics"]["macro_f1_present_crops"],
                        "majority_baseline": p["majority_baseline_macro_f1"], "meets_f1_criteria": p["meets_f1_criteria"],
                        **{f"{c}_f1": v["f1"] for c, v in p["metrics"]["per_crop"].items() if v["support"]}})
comparison = pd.DataFrame(records)
display(comparison.round(3))
display(comparison.pivot(index="population", columns="variant", values="present_crop_macro_f1").round(3))

## Reading the results

- The variants share most parcels (`shared`), so differences come from the few hundred parcels each
  side adds, mostly potato. Small differences are within seed noise; compare with the seed-stability
  spread from the tuning notebooks before reading them as real.
- Each variant chooses its own candidate on development data only. If the candidates differ, part
  of the score gap comes from the feature set, not the parcels.
- Preprocessing ran on the combined collection. A parcel removed there for sharing pixel locations
  with a parcel from the other collection stays removed in every variant.
- Software checks passing does not establish real-world crop accuracy.